#### **Python Type Hints for Pydantic**

Pydantic uses type hints to understand **what data your model expects** and then validates/parses that data.

---

#### **What are Type Hints?**

A type hint tells Python and developers what type of value a variable is expected to contain.

```python
name: str = "Shourov"
age: int = 25
height: float = 5.8
is_active: bool = True
```

Here:

| Variable    | Type Hint | Value       |
| ----------- | --------- | ----------- |
| `name`      | `str`     | `"Shourov"` |
| `age`       | `int`     | `25`        |
| `height`    | `float`   | `5.8`       |
| `is_active` | `bool`    | `True`      |

Type hints don't normally enforce types by themselves.
```python
age: int = "hello"
```
Python will generally allow this at runtime, even though it violates the type hint. That's where tools like **Pydantic** become important.

---

#### **Why Pydantic Uses Type Hints**

Consider:

In [1]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

The type hints tell Pydantic:

```text
name → should be a string
age  → should be an integer
```

In [2]:
user = User(name='Shourov Roy', age=23)
print(user)

name='Shourov Roy' age=23


Pydantic uses those annotations for **validation and data parsing**.

---

#### **Basic Python Types**

* `str`, `int`, `float`, `bool`: 

```python
class User(BaseModel):
    name: str
    age: int
    price: float
    is_active: bool    
```
---

#### **Lists**

Python type hints can describe collections.

```python
from typing import List
numbers: List[int]
```

For modern Pydantic projects, prefer:

```python
class Student(BaseModel):
    marks: list[int]
```
**Example:**

In [7]:
from pydantic import BaseModel

class Student(BaseModel):
    marks: list[int]

student = Student(marks=[60, 70, 80, 90])
print(student)

marks=[60, 70, 80, 90]


---

#### **Dictionaries**

You can specify the key and value types.

```python
data: dict[str, int]
```

**Example:**

In [8]:
from pydantic import BaseModel

class Student(BaseModel):
    scores: dict[str, int]

student = Student(
    scores={
        "Python": 4,
        "DSA": 4
    }
)
print(student)

scores={'Python': 4, 'DSA': 4}


---

#### **Optional Values**

Sometimes a field may contain a value or `None`. Modern Python:

```python
name: str | None = None
```
**Example:**

In [11]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    nickname: str | None = None

user1 = User(name="Shourov")
print(user1)

user2 = User(name="Shourov", nickname="Roy")
print(user2)

name='Shourov' nickname=None
name='Shourov' nickname='Roy'


The important distinction is:

```python
str | None
```

means the value **can be `None`**.

---

#### **Default Values**

You can provide defaults:

In [12]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int = 20

user = User(name="Shourov Roy")
print(user.age)

20


---

#### **Nested Type Hints**

This becomes very useful with Pydantic.

In [15]:
from pydantic import BaseModel

class Address(BaseModel):
    city: str
    country: str

class User(BaseModel):
    name: str
    address: Address

user = User(
    name= "Shourov Roy",
    address = {
        "city": "Dinajpur",
        "country": "Bangladesh"
    }
)

print(user.name)
print(user.address)

Shourov Roy
city='Dinajpur' country='Bangladesh'


Pydantic can parse the nested dictionary into an `Address` model. Conceptually:

```text
User
 ├── name: str
 └── address: Address
       ├── city: str
       └── country: str
```

This concept becomes extremely important when you use **Pydantic with FastAPI**.

---

`Literal`: Sometimes you want to restrict a field to specific values.

In [20]:
from pydantic import BaseModel
from typing import Literal

class User(BaseModel):
    role: Literal["admin", "user"]

user1 = User(role="admin")
user2 = User(role="user")
print(user1)
print(user2)

# user3 = User(role="manager") this code shows error

role='admin'
role='user'


This is useful when your API accepts a fixed set of choices.

---

`Union`: A value can sometimes have multiple possible types. Older style:

```python
from typing import Union
value: Union[int, str]
```

Modern Python:

```python
value: int | str
```

**Example:**

In [22]:
from pydantic import BaseModel

class Product(BaseModel):
    product_id: int | str

p1 = Product(product_id=101)
p2 = Product(product_id="ABC101")
print(p1)
print(p2)

product_id=101
product_id='ABC101'


---

`Annotated`: It becomes very important when working with **Pydantic and FastAPI**.

```python
from typing import Annotated
```

It allows you to attach additional metadata or constraints to a type. For example:

In [25]:
from typing import Annotated
from pydantic import BaseModel, Field

class User(BaseModel):
    age: Annotated[int, Field(ge=18)]

user = User(age = 25)
print(user.age)

# user = User(age = 15) this shows error. this fails to validate

25


---

#### **Type Hints + Pydantic Validation**

This is the key idea to understand. Without Pydantic:

```python
age: int = "25"
```

The annotation itself doesn't perform runtime validation. With Pydantic:

In [27]:
from pydantic import BaseModel

class User(BaseModel):
    age: int

user = User(age = 25)
print(user.age)

25


Pydantic can parse the input into `25`. So Pydantic is doing more than simply reading the annotation.

---

#### **Type Hints + FastAPI**

```python
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class User(BaseModel):
    name: str
    age: int

@app.post("/users")
def create_user(user: User):
    return user
```
The annotation `user: User` tells FastAPI that the request body should be validated against the `User` Pydantic model. Request:

```json
{
    "name": "Shourov",
    "age": 25
}
```

FastAPI → Pydantic → validation → Python object. Conceptually:

```text
JSON Request
     ↓
FastAPI
     ↓
Pydantic Model
     ↓
Type Validation
     ↓
Python Object
```

---
#### **Advanced Type Hints**

```python
TypeVar
Generic
Protocol
TypedDict
Callable
TypeAlias
NewType
```
---

#### **A Small Practice Model**

In [29]:
from typing import Annotated, Literal
from pydantic import BaseModel, Field

# Nested model
class Address(BaseModel):
    city: str
    country: str

# Main model
class User(BaseModel):
    username: str
    age: Annotated[int, Field(ge=18)]
    email: str
    roles: list[str]
    address: Address
    status: Literal["active", "inactive"]
    nickname: str | None = None
    
# Creating a user
user1 = User(
    username="shourov",
    age=23,
    email="shourov@example.com",
    roles=["student", "developer"],
    address={
        "city": "Sylhet",
        "country": "Bangladesh"
    },
    status="active"
)
print(user1)

username='shourov' age=23 email='shourov@example.com' roles=['student', 'developer'] address=Address(city='Sylhet', country='Bangladesh') status='active' nickname=None
